# 04 · Friction Analysis — Peras / PostHog

**Purpose:** Trace individual sessions in chronological order to find where users
stop — the last meaningful event before a session ends. Exports two Tableau-ready
tables for internal friction-point analysis:
- `session_trace.csv` — event-level, one row per event, chronological within session.
- `session_summary.csv` — one row per session, with `last_meaningful_event` as an
  actual column.

**Safety contract for this notebook**
- Reads `data/raw/events_clean.parquet` (private tier, gitignored).
- **This notebook's exports are private-tier, not the public portfolio.** Unlike
  `03_eda.ipynb`, nothing here is aggregated or small-cell-suppressed — a
  session-by-session trace is inherently person-level data. Both output CSVs go to
  `data/raw/` (gitignored, and `*.csv` is gitignored repo-wide too) for **Tableau
  Desktop, opened locally** — not Tableau Public, and never committed.
- No cell in this notebook displays a raw session/person-level row. Only
  shape/dtype/aggregate-count outputs, per `CLAUDE.md`'s pre-commit safety
  checklist — notebook outputs get committed even though the data files don't.

**Before running:** run `01_extract.ipynb` then `02_clean_audit.ipynb` first so
`data/raw/events_clean.parquet` exists.


In [ ]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
IN_PATH = RAW_DIR / "events_clean.parquet"

assert IN_PATH.exists(), f"{IN_PATH} not found -- run 01_extract.ipynb and 02_clean_audit.ipynb first."

events = pd.read_parquet(IN_PATH)
events["timestamp"] = pd.to_datetime(events["timestamp"], utc=True, format="ISO8601")

print(f"Loaded {len(events):,} rows, {events['person_id'].nunique():,} people, "
      f"{events['session_id'].nunique():,} sessions.")

## Step 1 · Bot filter (defensive re-check)

`events_clean.parquet` already had bots and non-production traffic removed in
`02_clean_audit.ipynb`. This step re-asserts that explicitly rather than trusting
it silently — if this notebook is ever pointed at a different input, or the
upstream filter changes, this still guarantees a bot-free dataframe.


In [ ]:
n_before = len(events)
still_bot_like = events["is_bot"].fillna(False).astype(bool) | (
    events["traffic_type"].notna() & (events["traffic_type"] != "Regular")
)
events_bf = events.loc[~still_bot_like].reset_index(drop=True)

n_removed = n_before - len(events_bf)
print(f"Removed {n_removed:,} additional bot/non-regular-traffic rows ({n_removed / n_before:.2%}).")
print("Expected ~0 here -- events_clean.parquet should already be bot-free from "
      "02_clean_audit.ipynb; this is a defensive re-check, not the primary filter.")
print(f"Bot-free rows: {len(events_bf):,}")

## Step 2 · Session sequencing (chronological order + position within session)

Sessions are PostHog's `session_id` (client-side, ~30 min inactivity timeout --
same definition used in `02_clean_audit.ipynb`). Rows with no `session_id`
(server-side events with no browser session context) can't be placed in a session
trace, so they're dropped here specifically -- they're still in
`events_clean.parquet` for other analyses, just not traceable session-by-session.


In [ ]:
n_before_session_filter = len(events_bf)
events_bf = events_bf.loc[events_bf["session_id"].notna()].copy()
n_dropped_no_session = n_before_session_filter - len(events_bf)
print(f"Dropped {n_dropped_no_session:,} rows with no session_id "
      f"({n_dropped_no_session / n_before_session_filter:.2%}) -- not traceable to a session.")

events_bf = events_bf.sort_values(["session_id", "timestamp"]).reset_index(drop=True)

events_bf["session_seq"] = events_bf.groupby("session_id").cumcount() + 1
events_bf["session_event_count"] = events_bf.groupby("session_id")["session_id"].transform("size")
events_bf["is_last_in_session"] = events_bf["session_seq"] == events_bf["session_event_count"]

print(f"Sessions: {events_bf['session_id'].nunique():,}")
print(f"Events per session -- median: {events_bf['session_event_count'].median():.0f}, "
      f"max: {events_bf['session_event_count'].max():,}")

## Step 3 · Meaningful-event flagging

Not every event reflects a real action — `$feature_flag_called`/`$web_vitals` fire
automatically alongside real activity, and `$identify`/`$create_alias`/`$set` are
identity plumbing, not something a user "did." Excluding these (rather than
including only a hand-picked allowlist) means real product events don't need to be
re-added here every time Peras ships a new one. Deliberately **not** excluding
`$exception` or `$rageclick` — an error or a rage-click is exactly the kind of
friction signal this notebook exists to surface, not noise to hide.

`NOISE_EVENTS` is tunable — adjust if a specific analysis needs a different
definition of "meaningful."


In [ ]:
NOISE_EVENTS = {
    "$feature_flag_called",
    "$web_vitals",
    "$identify",
    "$create_alias",
    "$set",
}

events_bf["is_meaningful"] = ~events_bf["event"].isin(NOISE_EVENTS)

meaningful_share = events_bf["is_meaningful"].mean()
print(f"Meaningful events: {meaningful_share:.1%} of all rows "
      f"({len(NOISE_EVENTS)} event types excluded as noise: {sorted(NOISE_EVENTS)}).")

## Step 4 · Last-meaningful-event-per-session flagging

This is the core friction signal: for each session, which row is the *last
meaningful* event before the session ends? `is_last_in_session` (Step 2) marks the
literal last row regardless of whether it's noise; `is_last_meaningful_in_session`
marks the last row that was actually a real action -- the more useful one for
"what made them stop."


In [ ]:
last_meaningful_seq = (
    events_bf.loc[events_bf["is_meaningful"]]
    .groupby("session_id")["session_seq"]
    .max()
    .rename("last_meaningful_seq")
)
events_bf = events_bf.merge(last_meaningful_seq, on="session_id", how="left")
events_bf["is_last_meaningful_in_session"] = (
    events_bf["is_meaningful"] & (events_bf["session_seq"] == events_bf["last_meaningful_seq"])
)
events_bf = events_bf.drop(columns="last_meaningful_seq")

n_sessions_no_meaningful = (~events_bf.groupby("session_id")["is_meaningful"].any()).sum()
print(f"Sessions with zero meaningful events: {n_sessions_no_meaningful:,} "
      f"(pure noise/instrumentation only -- no is_last_meaningful_in_session flag possible for these).")

## Step 5 · Session timing (start, end, duration, date)

Session-level timing, joined back onto every row in that session so each event
carries its own session's context. `session_date` is a plain date (no time
component) for coarse day-level filtering in Tableau, alongside the full
`session_start`/`session_end` timestamps for exact-range filtering -- e.g. to
align with when a specific experiment was running.


In [ ]:
session_timing = (
    events_bf.groupby("session_id")["timestamp"]
    .agg(session_start="min", session_end="max")
    .reset_index()
)
session_timing["session_duration_seconds"] = (
    session_timing["session_end"] - session_timing["session_start"]
).dt.total_seconds()
session_timing["session_date"] = session_timing["session_start"].dt.date

events_bf = events_bf.merge(session_timing, on="session_id", how="left")

print(f"Session duration (seconds) -- median: {events_bf['session_duration_seconds'].median():.0f}, "
      f"max: {events_bf['session_duration_seconds'].max():,.0f}")

## Step 6 · Person-level context (channel, experiments, signup/activation)

Same first-touch approach as `03_eda.ipynb`: each person's first-assigned channel
and experiment variant, plus whether they ever signed up / activated -- joined onto
every row so friction can be sliced by acquisition channel or experiment arm
directly in Tableau, not just by date range. Prefixed `person_*` to avoid colliding
with the existing sparse event-level `landing_variant`/`campaign_variant`/
`sample_consent_variant` columns (those stay as-is, showing only the row where the
assignment actually fired); the `person_*` versions are back-filled onto every row
for that person.


In [ ]:
def first_touch(df, col):
    d = df.dropna(subset=[col]).sort_values("timestamp")
    return d.groupby("person_id")[col].first()

first_utm_source = first_touch(events_bf, "utm_source")
first_referring_domain = first_touch(events_bf, "referring_domain")

person_context = pd.DataFrame(index=events_bf["person_id"].unique())
person_context.index.name = "person_id"
person_context["person_channel"] = first_utm_source
person_context["person_channel"] = person_context["person_channel"].fillna(first_referring_domain)
person_context["person_channel"] = person_context["person_channel"].fillna("direct/unknown")

person_context["person_landing_variant"] = first_touch(events_bf, "landing_variant")
person_context["person_campaign_variant"] = first_touch(events_bf, "campaign_variant")
person_context["person_sample_consent_variant"] = first_touch(events_bf, "sample_consent_variant")

SIGNUP_EVENTS = ["account.signed_up"]
ACTIVATION_EVENTS = ["generation.completed", "generation.first_completed"]
signed_up_people = set(events_bf.loc[events_bf["event"].isin(SIGNUP_EVENTS), "person_id"])
activated_people = set(events_bf.loc[events_bf["event"].isin(ACTIVATION_EVENTS), "person_id"])
person_context["person_signed_up"] = person_context.index.isin(signed_up_people)
person_context["person_activated"] = person_context.index.isin(activated_people)

person_context = person_context.reset_index()
events_bf = events_bf.merge(person_context, on="person_id", how="left")

print(f"Person-level context joined for {len(person_context):,} people.")

## Step 7 · Build the session-level summary table

One row per session -- the compact view for a quick "what's the most common last
meaningful event" chart, without needing to filter the full event-level trace in
Tableau first.


In [ ]:
last_meaningful_event = (
    events_bf.loc[events_bf["is_last_meaningful_in_session"], ["session_id", "event"]]
    .rename(columns={"event": "last_meaningful_event"})
)
last_event_overall = (
    events_bf.loc[events_bf["is_last_in_session"], ["session_id", "event"]]
    .rename(columns={"event": "last_event_overall"})
)
session_person = events_bf.groupby("session_id").agg(
    person_id=("person_id", "first"),
    session_event_count=("session_event_count", "first"),
).reset_index()

session_summary = (
    session_timing
    .merge(session_person, on="session_id", how="left")
    .merge(last_meaningful_event, on="session_id", how="left")
    .merge(last_event_overall, on="session_id", how="left")
    .merge(person_context, on="person_id", how="left")
)

print(f"Session summary: {len(session_summary):,} rows (one per session).")
print("Most common last meaningful event (aggregate counts only, no session/person IDs):")
display(session_summary["last_meaningful_event"].value_counts().head(15))

## Step 8 · Export for Tableau

Both files are private-tier (`data/raw/`, gitignored) for **Tableau Desktop opened
locally** -- not Tableau Public, and never committed. Timestamp columns
(`timestamp`, `session_start`, `session_end`) stay as real pandas `datetime64`
values right up until export, so `to_csv` writes clean ISO8601 strings that Tableau
auto-detects as a Date & Time field on import.


In [ ]:
trace_path = RAW_DIR / "session_trace.csv"
summary_path = RAW_DIR / "session_summary.csv"

events_bf.to_csv(trace_path, index=False)
session_summary.to_csv(summary_path, index=False)

print(f"Wrote {len(events_bf):,} rows -> {trace_path.resolve()}")
print(f"Wrote {len(session_summary):,} rows -> {summary_path.resolve()}")
print("Reminder: data/raw/ is gitignored (and *.csv is gitignored repo-wide too). "
      "These files stay private -- for Tableau Desktop opened locally, not Tableau Public.")

## Data Dictionary

Two layers, generated from one dictionary below so they can't drift apart from
each other: a coarse one (what each column *means*) and a detailed one (exactly
how it was derived, and which step above to look at for the code).


In [ ]:
COLUMN_DICTIONARY = {
    "event": {
        "table": "both", "coarse": "The event name (e.g. $pageview, generation.completed).",
        "detail": "Pulled directly from PostHog's events table in 01_extract.ipynb. In session_summary this only appears as last_meaningful_event / last_event_overall, not as its own column.",
    },
    "timestamp": {
        "table": "trace", "coarse": "When this specific event happened (UTC).",
        "detail": "events.timestamp from 01_extract.ipynb, parsed with pd.to_datetime(..., format='ISO8601') in this notebook's load cell to handle PostHog's mixed fractional-second precision.",
    },
    "distinct_id": {
        "table": "trace", "coarse": "PostHog's raw client-side identifier for this event.",
        "detail": "Pulled in 01_extract.ipynb. Can differ across a single person's anonymous-to-identified transition -- person_id is the canonical identity, this is kept for low-level debugging only.",
    },
    "person_id": {
        "table": "both", "coarse": "The canonical person identifier -- use this to trace one person across sessions.",
        "detail": "events.person_id from 01_extract.ipynb -- merges anonymous and identified distinct_ids for the same person.",
    },
    "account_id": {
        "table": "trace", "coarse": "Peras's own internal account identifier (not PostHog's).",
        "detail": "properties.account_id, pulled in 01_extract.ipynb. Sparse -- only present on events that carry it server-side.",
    },
    "project_id": {
        "table": "trace", "coarse": "Which textbook/project this event belongs to, if any.",
        "detail": "properties.project_id, pulled in 01_extract.ipynb. Sparse -- only present on project/generation events.",
    },
    "session_id": {
        "table": "both", "coarse": "PostHog's session identifier -- groups events into one browsing session.",
        "detail": "properties.$session_id, pulled in 01_extract.ipynb. This notebook's Step 2 drops any row where it's null (not traceable to a session) and uses it as the grouping key for every session_* column below.",
    },
    "session_seq": {
        "table": "trace", "coarse": "This event's position within its session (1, 2, 3, ...), in chronological order.",
        "detail": "Step 2 -- Session sequencing: events_bf.groupby('session_id').cumcount() + 1 after sorting by (session_id, timestamp).",
    },
    "session_event_count": {
        "table": "both", "coarse": "Total number of events in this session.",
        "detail": "Step 2 -- Session sequencing: events_bf.groupby('session_id')['session_id'].transform('size').",
    },
    "is_last_in_session": {
        "table": "trace", "coarse": "True if this is the last event in the session, regardless of what it is.",
        "detail": "Step 2 -- Session sequencing: session_seq == session_event_count.",
    },
    "is_meaningful": {
        "table": "trace", "coarse": "True if this event is a real action, not automatic instrumentation.",
        "detail": "Step 3 -- Meaningful-event flagging: event not in NOISE_EVENTS (see that step for the exact excluded list and rationale).",
    },
    "is_last_meaningful_in_session": {
        "table": "trace", "coarse": "True if this is the last *meaningful* event in the session -- the friction-point row.",
        "detail": "Step 4 -- Last-meaningful-event-per-session flagging: is_meaningful AND session_seq equals the max session_seq among meaningful rows in that session.",
    },
    "session_start": {
        "table": "both", "coarse": "Timestamp of the first event in the session.",
        "detail": "Step 5 -- Session timing: events_bf.groupby('session_id')['timestamp'].min().",
    },
    "session_end": {
        "table": "both", "coarse": "Timestamp of the last event in the session.",
        "detail": "Step 5 -- Session timing: events_bf.groupby('session_id')['timestamp'].max().",
    },
    "session_duration_seconds": {
        "table": "both", "coarse": "How long the session lasted, in seconds.",
        "detail": "Step 5 -- Session timing: (session_end - session_start).dt.total_seconds().",
    },
    "session_date": {
        "table": "both", "coarse": "The calendar date the session started (no time component) -- for coarse day-level filtering.",
        "detail": "Step 5 -- Session timing: session_start.dt.date.",
    },
    "current_url": {
        "table": "trace", "coarse": "The page URL at the time of this event (query string stripped).",
        "detail": "properties.$current_url from 01_extract.ipynb, with splitByChar('?', ...) applied at extraction to strip ad-click-ID query params -- see that notebook's PII checklist.",
    },
    "browser": { "table": "trace", "coarse": "Browser name.", "detail": "properties.$browser, pulled in 01_extract.ipynb." },
    "os": { "table": "trace", "coarse": "Operating system.", "detail": "properties.$os, pulled in 01_extract.ipynb." },
    "country": { "table": "trace", "coarse": "Country, from IP geolocation (coarse only -- no city/postal/lat-long).", "detail": "properties.$geoip_country_code, pulled in 01_extract.ipynb." },
    "referring_domain": {
        "table": "trace", "coarse": "Domain the visitor arrived from, if any (e.g. google.com).",
        "detail": "properties.$referring_domain, pulled in 01_extract.ipynb. Also used as a fallback for person_channel in this notebook's Step 6 when utm_source is missing.",
    },
    "entry_path": { "table": "trace", "coarse": "The page/flow entry point.", "detail": "properties.entry_path, pulled in 01_extract.ipynb." },
    "auth_provider": { "table": "trace", "coarse": "Which auth method was used to sign up (e.g. Google, email).", "detail": "properties.auth_provider, pulled in 01_extract.ipynb. Only present on account.signed_up." },
    "utm_source": { "table": "trace", "coarse": "UTM source tag from marketing links.", "detail": "properties.utm_source, pulled in 01_extract.ipynb." },
    "utm_medium": { "table": "trace", "coarse": "UTM medium tag.", "detail": "properties.utm_medium, pulled in 01_extract.ipynb." },
    "utm_campaign": { "table": "trace", "coarse": "UTM campaign tag.", "detail": "properties.utm_campaign, pulled in 01_extract.ipynb." },
    "utm_term": { "table": "trace", "coarse": "UTM term tag.", "detail": "properties.utm_term, pulled in 01_extract.ipynb." },
    "landing_experiment": { "table": "trace", "coarse": "Which landing-page experiment this specific row's event belongs to, if it's the assignment event.", "detail": "properties.landing_experiment, pulled in 01_extract.ipynb. Sparse -- only set on landing_experiment_assigned rows. See person_landing_variant for a per-person filled version." },
    "landing_variant": { "table": "trace", "coarse": "Which landing-page variant was assigned, on the assignment event row only.", "detail": "properties.landing_variant, pulled in 01_extract.ipynb. Sparse -- see person_landing_variant (Step 6) for the back-filled version." },
    "campaign_intent": { "table": "trace", "coarse": "Marketing campaign intent property.", "detail": "properties.campaign_intent, pulled in 01_extract.ipynb. Sparse." },
    "campaign_variant": { "table": "trace", "coarse": "Which campaign-experiment variant, on the assignment event row only.", "detail": "properties.campaign_variant, pulled in 01_extract.ipynb. Sparse -- see person_campaign_variant (Step 6) for the back-filled version." },
    "sample_consent_experiment": { "table": "trace", "coarse": "Which onboarding-consent experiment, on the assignment event row only.", "detail": "properties.sample_consent_experiment, pulled in 01_extract.ipynb. Sparse." },
    "sample_consent_variant": { "table": "trace", "coarse": "Which onboarding-consent variant, on the assignment event row only.", "detail": "properties.sample_consent_variant, pulled in 01_extract.ipynb. Sparse -- see person_sample_consent_variant (Step 6) for the back-filled version." },
    "generation_mode": { "table": "trace", "coarse": "Which textbook-generation tool/mode was used.", "detail": "properties.generation_mode, pulled in 01_extract.ipynb. Sparse -- only present on generation events." },
    "reader_mode": { "table": "trace", "coarse": "Reader UI mode.", "detail": "properties.reader_mode, pulled in 01_extract.ipynb. Sparse -- only present on reader events." },
    "reader_section_count": { "table": "trace", "coarse": "Number of sections in the textbook being read.", "detail": "properties.reader_section_count, pulled in 01_extract.ipynb. Sparse." },
    "reader_written_sections": { "table": "trace", "coarse": "Number of sections written so far.", "detail": "properties.reader_written_sections, pulled in 01_extract.ipynb. Sparse." },
    "has_paced_reader": { "table": "trace", "coarse": "Whether the paced-reader feature was active.", "detail": "properties.has_paced_reader, pulled in 01_extract.ipynb. Sparse." },
    "environment": { "table": "trace", "coarse": "production vs. other -- should be uniform post-filter, kept for audit transparency.", "detail": "properties.environment, pulled in 01_extract.ipynb; non-production rows were removed in 02_clean_audit.ipynb, so this should read 'production' or null here." },
    "is_bot": { "table": "trace", "coarse": "PostHog's bot classification -- should be entirely False in this export.", "detail": "properties.$virt_is_bot, pulled in 01_extract.ipynb; bot rows were removed in 02_clean_audit.ipynb and re-checked in this notebook's Step 1. Kept for audit transparency, not because it varies." },
    "traffic_type": { "table": "trace", "coarse": "PostHog's traffic classification -- should be 'Regular' or null in this export.", "detail": "properties.$virt_traffic_type, pulled in 01_extract.ipynb; non-Regular rows were removed in 02_clean_audit.ipynb and re-checked in this notebook's Step 1. Kept for audit transparency." },
    "person_channel": { "table": "both", "coarse": "This person's first-touch acquisition channel (constant across all their rows).", "detail": "Step 6 -- Person-level context: first non-null utm_source, falling back to referring_domain, falling back to 'direct/unknown', per person -- same logic as 03_eda.ipynb's acquisition analysis." },
    "person_landing_variant": { "table": "both", "coarse": "This person's assigned landing-page variant (constant across all their rows).", "detail": "Step 6 -- Person-level context: first non-null landing_variant per person, back-filled onto every row for that person." },
    "person_campaign_variant": { "table": "both", "coarse": "This person's assigned campaign variant (constant across all their rows).", "detail": "Step 6 -- Person-level context: first non-null campaign_variant per person, back-filled." },
    "person_sample_consent_variant": { "table": "both", "coarse": "This person's assigned consent-experiment variant (constant across all their rows).", "detail": "Step 6 -- Person-level context: first non-null sample_consent_variant per person, back-filled." },
    "person_signed_up": { "table": "both", "coarse": "Whether this person ever signed up (account.signed_up), anywhere in the data.", "detail": "Step 6 -- Person-level context: person_id is in the set of people with an account.signed_up event." },
    "person_activated": { "table": "both", "coarse": "Whether this person ever activated (generated a textbook), anywhere in the data.", "detail": "Step 6 -- Person-level context: person_id is in the set of people with a generation.completed or generation.first_completed event." },
    "last_meaningful_event": { "table": "summary", "coarse": "The name of the last meaningful event in this session -- the friction-point signal.", "detail": "Step 7 -- Session summary: event value from the trace row where is_last_meaningful_in_session is True, joined onto session_summary." },
    "last_event_overall": { "table": "summary", "coarse": "The name of the literal last event in this session, regardless of meaningfulness.", "detail": "Step 7 -- Session summary: event value from the trace row where is_last_in_session is True." },
}

coarse_dict = pd.DataFrame(
    [{"column": c, "in": v["table"], "description": v["coarse"]} for c, v in COLUMN_DICTIONARY.items()]
)
coarse_dict

In [ ]:
detailed_dict = pd.DataFrame(
    [{"column": c, "derivation": v["detail"]} for c, v in COLUMN_DICTIONARY.items()]
)
with pd.option_context("display.max_colwidth", None):
    display(detailed_dict)

---

## Notes on using this in Tableau

- **Timeframe filtering:** use `session_date` for coarse day-level filters, or
  `session_start`/`session_end` for exact ranges -- both are real datetime fields,
  not strings, so Tableau should auto-detect them as Date & Time on import. If it
  doesn't, set the field type manually in the data source pane.
- **Aligning with experiments:** filter by `person_landing_variant` /
  `person_campaign_variant` / `person_sample_consent_variant` directly, rather than
  only inferring experiment windows from dates.
- **Finding friction points:** filter `session_trace.csv` to
  `is_last_meaningful_in_session = True` and chart `event` -- or use
  `session_summary.csv`'s `last_meaningful_event` column directly, which is the
  same information pre-aggregated to one row per session.
- **This is private data.** Both CSVs live in `data/raw/`, gitignored, for local
  Tableau Desktop use -- do not upload to Tableau Public or any other public
  destination without going through the same aggregation/suppression/clearance
  process `03_eda.ipynb` and `CLAUDE.md` require for the public tier.

## Next

Findings from this analysis feed into `reports/findings_memo.md` alongside
`03_eda.ipynb`'s results -- both are inputs to the internal "for Peras" deliverable,
not the public portfolio piece.
